# 3 · Class nodes and what a run supplies

Every node is a `NodeDefinition` subclass, so a node that needs helpers keeps them on the class. The engine makes a fresh instance for every call: a node keeps nothing between calls.

Some nodes need something only the caller of `execute` has — who is running the graph, a clock, a client for a service. That is not a value any node produces or a person types, so it is not an input. The parameter says `Annotated[T, FromRun()]`, and the caller hands the value to `execute(from_run={T: value})`.

In [ ]:
from typing import Annotated

from conductor import CompiledGraph, Edges, FromRun, Graph, GraphNode, NodeDefinition, NodeRegistry, Param, Ref, Result, Static
from conductor.execution.engine import collect, execute
from conductor.interface import model_of
from conductor.widgets import Textarea
from conductor_nodes.types import Number, Text

registry = NodeRegistry()

## A class node

Helpers live on the class; `run` is the interface. A node is plain Python, so it can be called directly.

In [ ]:
class WordCount(NodeDefinition):
    id = "word-count"
    title = "Word count"
    description = "Counts the words in a text"
    category = "text"

    def run(self, text: Annotated[Text, Param(title="Text", widget=Textarea())]) -> Annotated[Number, Result(title="Words")]:
        return Number(len(self._words(text)))

    @staticmethod
    def _words(text: str) -> list[str]:
        return text.split()


registry.register(WordCount)
WordCount().run(text=Text("the quick brown fox"))

## A value the run supplies

`Caller` is a type the host owns. `who: Annotated[Caller, FromRun()]` says the value is handed in by whoever runs the graph: no widget, no handle, nothing to bind.

In [ ]:
class Caller:
    """Whoever is running the graph — a type the host declares."""

    def __init__(self, name: str) -> None:
        self.name = name


class Signed(NodeDefinition):
    id = "signed"
    title = "Signed"
    description = "Signs a text with the name of whoever ran the graph"
    category = "text"

    def run(
        self,
        text: Annotated[Text, Param(title="Text", widget=Textarea())],
        who: Annotated[Caller, FromRun()],
    ) -> Annotated[Text, Result(title="Signed")]:
        return Text(f"{text} — {who.name}")


registry.register(Signed)

## It is a need, not an input

`Interface.needs` names it by parameter; `Interface.inputs`, and the model a call is validated through, do not know it exists.

In [ ]:
interface = Signed.versions[1].interface

print("inputs:", [inp.name for inp in interface.inputs])
print("needs: ", {name: needed.__name__ for name, needed in interface.needs.items()})
assert "who" not in model_of(interface.inputs).model_fields

## Run it

The caller hands the value in by type. A node receives only what its own signature names; there is no context object.

In [ ]:
compiled = CompiledGraph.from_graph(
    Graph(nodes=[
        GraphNode(id="note", type="signed", version=1, bindings={"text": Static(value="The quick brown fox jumps over the lazy dog")}),
        GraphNode(id="count", type="word-count", version=1, bindings={"text": Edges(refs=(Ref("note", "result"),))}),
    ]),
    registry,
)

results = await collect(execute(compiled, from_run={Caller: Caller("Ida")}))
print(results["note"]["result"])
print(results["count"]["result"])

A leg that is not given what its nodes need refuses to start, before any node runs:

In [ ]:
try:
    await collect(execute(compiled))
except TypeError as refused:
    print(refused)